In [ ]:
import os
os.chdir("../../")

In [ ]:
!pwd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

from pyomo2h5 import load_yaml
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.style.use("FST_bw.mplstyle")

## Load approximation and exact coefficients
load from optproblems.yaml

In [ ]:
data = load_yaml("opt_problems/preplanning/GPZ/standard_case.yml")

### Linearisierung logarithmus

In [ ]:
x_approx = {
            1: 0.001,
            2: 0.0021544346900318843,
            3: 0.004641588833612777,
            4: 0.01,
            5: 0.021544346900318832,
            6: 0.046415888336127774,
            7: 0.1,
            8: 0.21544346900318823,
            9: 0.46415888336127775,
            10: 1.0,
        }

fx_approx = {
            1: -3.0,
            2: -2.6666666666666665,
            3: -2.3333333333333335,
            4: -2.0,
            5: -1.6666666666666667,
            6: -1.3333333333333335,
            7: -1.0,
            8: -0.666666666666667,
            9: -0.3333333333333335,
            10: 0.0,
        }

def max_log10_error(x: float) -> float:
    """
    Berechnet den maximalen vertikalen Abstand zwischen log10(x)
    und der stückweise linearen Interpolation durch die Stützpunkte.
    """

    if x <= 0:
        raise ValueError("x muss positiv sein.")

    keys = sorted(x_approx)

    if x < x_approx[keys[0]] or x > x_approx[keys[-1]]:
        raise ValueError(
            f"x muss im Bereich [{x_approx[keys[0]]}, {x_approx[keys[-1]]}] liegen."
        )

    for k1, k2 in zip(keys[:-1], keys[1:]):
        x1, x2 = x_approx[k1], x_approx[k2]
        y1, y2 = fx_approx[k1], fx_approx[k2]

        if x1 <= x <= x2:
            y_lin = y1 + (y2 - y1) * (x - x1) / (x2 - x1)
            y_exact = math.log10(x)
            return abs(y_lin - y_exact)

    return 0.0

Plot approximation of logarithm

In [ ]:
x = np.logspace(-3,0,100)

plt.plot(x_approx.values(), fx_approx.values())
plt.plot(x, np.log10(x))


plot error of linearisation

In [ ]:
plt.plot(x,[20*max_log10_error(xi) for xi in x])

In [ ]:
print(f"Max error is {max(20*max_log10_error(xi) for xi in x)}")

## linearisation of the fan octave correction

In [ ]:
f_m =  np.array([63, 125, 250, 500, 1000, 2000, 4000, 8000])

def pegel_add(*L):
    return 10 * np.log10(np.sum([10 ** (0.1 * x) for x in L]))

def fan_octave_sounds(n):
    K1 = 0.4

    Strouhal = f_m * 60 / np.pi / n
    Delta_L_W_Okt = -5 -5 * (np.log10(Strouhal)+K1)**2

    Delta_L_W_Okt_Gesamt = pegel_add(Delta_L_W_Okt)

    L_W_Okt = Delta_L_W_Okt - Delta_L_W_Okt_Gesamt
    return L_W_Okt

In [ ]:
fan_pressure_coefficients = data["fan_pressure_coefficients"]
fan_flow_noise_coefficients = data["fan_flow_noise_coefficients"]

fan_pressure_max = data["fan_pressure_max"]
fan_volume_flow_max = data["fan_volume_flow_max"]

fan_rot_speed_max = data["fan_rotational_speed_max"]

In [ ]:
fan = ["Fan1", 0.5]

q_max = fan_volume_flow_max[*fan]
dp_max = fan_pressure_max[*fan]

alpha = {key[-1]:value for key, value in fan_pressure_coefficients.items() if key[:-1] == tuple(fan)}
epsilon = {(key[-2:]):value for key, value in fan_flow_noise_coefficients.items() if key[:2] == tuple(fan)}

In [ ]:
nlam = (
        lambda q, dp: (
            -alpha[2] * q + np.sqrt(q**2 * (alpha[2] ** 2 - 4 * alpha[1] * alpha[3]) + 4 * alpha[3] * dp)
        )
        / 2 / alpha[3]
    )

dp_char = lambda q, n: alpha[1]*q**2 + alpha[2]*q*n + alpha[3]*n**2

approx_octave_correction = lambda fi, q, dp: epsilon[fi, 1] * q + epsilon[fi, 2] * dp + epsilon[fi, 3]

In [ ]:
results = {}

qi_values = np.logspace(-3, 0, 100)*q_max
dpi_values = np.logspace(-3, 0, 10)

for fi in range(1, 9):
    results[fi] = {}

    for dpi in dpi_values:
        qi_list = []
        approx_list = []
        exact_list = []
        deviation_list = []

        for qi in qi_values:
            n = nlam(qi,dpi*dp_max)
            # print(n)
            if n < 0 or n > 1:
                continue
            # print(n)
            approx = approx_octave_correction(fi, qi*q_max, dpi*dp_max)
            exact = fan_octave_sounds(n * fan_rot_speed_max[*fan])[fi-1]
            deviation = approx - exact

            qi_list.append(qi)
            approx_list.append(approx)
            exact_list.append(exact)
            deviation_list.append(deviation)

        results[fi][dpi] = {
            "qi": np.array(qi_list),
            "approx": np.array(approx_list),
            "exact": np.array(exact_list),
            "deviation": np.array(deviation_list),
        }

    # print(np.max(list(deviation.values())))

In [ ]:
fi = 8


# for fi, val in results.items():
for dpi, values in results[fi].items():
    plt.plot(values["qi"], values["exact"],  color="red")

    plt.plot(values["qi"], values["approx"],  color="blue")
    # break

plt.xlabel("qi")
plt.ylabel("approx - exact")
# plt.legend()
